In [0]:
# Imports

from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window
import dateutil.parser
import requests
import pandas as pd

In [0]:
# Funções de verificação

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
# Preparação dos endereços das tabelas

catalog = "atividade_2"

bronze = "bronze"
silver = "silver"
gold = "gold"

bronze_schema = f"{catalog}.{bronze}"
silver_schema = f"{catalog}.{silver}"
gold_schema = f"{catalog}.{gold}"

landing_path = f"/Volumes/{catalog}/{bronze}/Landing"

print(catalog)
print(bronze_schema)
print(silver_schema)
print(gold_schema)
print(landing_path)

atividade_2
atividade_2.bronze
atividade_2.silver
atividade_2.gold
/Volumes/atividade_2/bronze/Landing


## Info Filmes

In [0]:
# Mapeamento tb_movies_info -> tb_info_filmes
silver_table_name = "tb_info_filmes"
df_info_filmes = spark.table(f"{bronze_schema}.tb_movies_info")

df_info_filmes = (
    df_info_filmes
        .where(F.col("id").isNotNull())
        .withColumnsRenamed({
            "id": "id_filme",
            "title": "titulo",
            "original_title": "titulo_original",
            "release_date": "data_lancamento",
            "runtime": "duracao_minutos",
            "original_language": "idioma_original",
            "status": "status_filme",
            "overview": "sinopse",
            "tagline": "frase_divulgacao"
        })
)


In [0]:
# Limpeza de status
df_info_filmes = (
    df_info_filmes.withColumn("status_filme",
                              F.upper(F.trim(F.regexp_replace(F.col("status_filme"), "-", " "))))
)

re_verify_status = "^(RELEASED|POST PRODUCTION|IN PRODUCTION|RUMORED|PLANNED|CANCELED)$"
standardized_condition = F.regexp(F.col("status_filme"), F.lit(re_verify_status))

dq_check(f"{silver_schema}.{silver_table_name}", "Verificação de status corrompidos", df_info_filmes, standardized_condition)

[FAIL] atividade_2.silver.tb_info_filmes | Verificação de status corrompidos | 36/962370 linhas falharam


In [0]:
df_corrupted_status = (
    df_info_filmes
        .where(~standardized_condition)
        .select("status_filme")
)

# Verificação manual dos status corrompidos
display(df_corrupted_status)

status_filme
HIS ERRANT DAD RETURNS
SOMETIMES REFERRED TO THE ISLAMIC RELIGIOUS POLICE. ALL THIS
"\CAN YOU STILL OPEN YOUR HEART TO A FRIEND WHO TURNS OUT TO BE """"\""""BOY A"""""
"FLOYD 'MONEY' MAYWEATHER PUTS ON A SHOW NO MATTER THE OPPONENT."""
HIS ERRANT DAD RETURNS
SOMETIMES REFERRED TO THE ISLAMIC RELIGIOUS POLICE. ALL THIS
"\CAN YOU STILL OPEN YOUR HEART TO A FRIEND WHO TURNS OUT TO BE """"\""""BOY A"""""
"FLOYD 'MONEY' MAYWEATHER PUTS ON A SHOW NO MATTER THE OPPONENT."""
HIS ERRANT DAD RETURNS
SOMETIMES REFERRED TO THE ISLAMIC RELIGIOUS POLICE. ALL THIS


In [0]:
# Tradução dos termos dos status para o português
df_info_filmes = (
    df_info_filmes.withColumn("status_filme", 
        F.when(F.col("status_filme") == "RELEASED", "Lançado")
            .when(F.col("status_filme") == "POST PRODUCTION", "Pós-Produção")
            .when(F.col("status_filme") == "IN PRODUCTION", "Em Produção")
            .when(F.col("status_filme") == "RUMORED", "Rumores")
            .when(F.col("status_filme") == "PLANNED", "Planejado")
            .when(F.col("status_filme") == "CANCELED", "Cancelado")
            .otherwise("Não Informado")
    )
)

df_distinct_status = (
    df_info_filmes
        .select("status_filme")
        .distinct()
)

dq_check(
    f"df_distinct_status", 
    "Verificação de status corrompidos após tratamento", 
    df_distinct_status, 
    F.col("status_filme").isin(
    ["Lançado", "Pós-Produção", "Em Produção", 
    "Rumores", "Planejado", "Cancelado", "Não Informado"]
    )
)

[PASS] df_distinct_status | Verificação de status corrompidos após tratamento | 0/5 linhas falharam


In [0]:
# Deduplicação de dados com base no ingestion_timestamp

ingestion_window = Window.partitionBy("id_filme").orderBy(F.desc("ingestion_datetime"))

df_info_filmes = (
    df_info_filmes
        .withColumn("row_number", F.row_number().over(ingestion_window))
        .where(F.col("row_number") == 1)
        .drop("row_number")
)

dq_check_unique(f"{silver_schema}.{silver_table_name}", "Verificação de duplicidade", df_info_filmes, ["id_filme"])

[PASS] atividade_2.silver.tb_info_filmes | Verificação de duplicidade | 0 chaves duplicadas de 97879 linhas


In [0]:
# Função para conversão em data a partir de lista de formatos
def to_date_formats(col, formats):
    return F.coalesce(*[F.try_to_date(col, f) for f in formats])

# Verificação de formatos de data
formats = ["dd/MM/yyyy", "dd-MM-yyyy", "MM/dd/yyyy", "MM-dd-yyyy", "yyyy/MM/dd", "yyyy-MM-dd"]
date_condition = to_date_formats(F.col("data_lancamento"), formats).isNotNull()
dq_check(f"{silver_schema}.{silver_table_name}", "Verificação de formatos de data", df_info_filmes, date_condition)

df_corrupted_date = (
    df_info_filmes.select("data_lancamento")
        .where(~date_condition)
)

# Verificação manual dos registros não comportados
display(df_corrupted_date)

[FAIL] atividade_2.silver.tb_info_filmes | Verificação de formatos de data | 64/97879 linhas falharam


data_lancamento
null
in partnership with brave rebel Saudi Arabian undercover reporters
his mother then committed suicide
null
null
null
at least for a brief moment
null
null
null


In [0]:
df_info_filmes = (
    df_info_filmes.withColumn("data_lancamento", 
        to_date_formats(F.col("data_lancamento"), formats))
)

In [0]:
# Verificação de tipos

display(df_info_filmes.dtypes)

_1,_2
id_filme,int
tconst,string
titulo,string
titulo_original,string
idioma_original,string
data_lancamento,date
duracao_minutos,string
status_filme,string
sinopse,string
frase_divulgacao,string


In [0]:
# Verificação de registros não numéricos

minute_type_check_condition = F.regexp(F.col("duracao_minutos"), F.lit("^[0-9]+$"))

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de tipos", 
    df_info_filmes,
    minute_type_check_condition)

display(
    df_info_filmes.select("duracao_minutos")
        .where(~minute_type_check_condition)
)

[FAIL] atividade_2.silver.tb_info_filmes | Verificação de tipos | 6/97879 linhas falharam


duracao_minutos
risk themselves under the laws of the land under the Committee for the Promotion of Virtue and the Prevention of Vice (CPVPV)
and he was left to grow up in an orphanage. Years later
"a boy who triggered his classmate's suicide."""
"loved."""
"start an eco-farm and put as much distance as he could between his family and the industrialization of nature. Until….. A wind park was planned at the gates of his paradise garden. Documentarian Poels and Alexander Pohl are taking the journey together…. to ask questions and unravel the green wonderland to its true core …"""
Conor McGregor


In [0]:
# Conversão de duração_minutos para inteiro

df_info_filmes = (
    df_info_filmes.withColumn("duracao_minutos", F.col("duracao_minutos").try_cast("int"))
)

dq_check(f"{silver_schema}.{silver_table_name}", 
    "Verificação de nulos", 
    df_info_filmes,
    F.col("duracao_minutos").isNotNull())

[FAIL] atividade_2.silver.tb_info_filmes | Verificação de nulos | 64/97879 linhas falharam


In [0]:
# Criação de coluna derivada e persistência da tabela na camada silver

df_info_filmes = (
    df_info_filmes.withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

df_info_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Financeiro Filmes

In [0]:
# Mapeamento tb_movies_financials -> tb_financeiro_filmes
silver_table_name = "tb_financeiro_filmes"
df_financeiro_filmes = spark.table(f"{bronze_schema}.tb_movies_financials")

df_financeiro_filmes = (
    df_financeiro_filmes
        .where(F.col("id").isNotNull())
        .dropDuplicates(["id"])
        .withColumnsRenamed({
            "budget": "orcamento_usd",
            "revenue": "receita_usd",
            "id": "id_filme"
        })
)

display(df_financeiro_filmes.limit(5))

id_filme,orcamento_usd,receita_usd,ingestion_datetime
293660,58000000,Unknown,2026-10-16T16:18:41.150Z
299536,300000000,2052415039,2026-10-16T16:18:41.150Z
299534,356000000,2800000000,2026-10-16T16:18:41.150Z
475557,55000000,1074458282,2026-10-16T16:18:41.150Z
271110,250000000,Não Informado,2026-10-16T16:18:41.150Z


In [0]:
# Verificação de tipos

display(df_financeiro_filmes.dtypes)

_1,_2
id_filme,int
orcamento_usd,string
receita_usd,string
ingestion_datetime,timestamp


In [0]:
def isNumerical(column):
    return F.regexp(F.col(column), F.lit(r"[\d\.]+"))

dq_check(f"{silver_schema}.{silver_table_name}", 
    "Verificação de orcamentos não numéricos", 
    df_financeiro_filmes,
    isNumerical("orcamento_usd")
)

dq_check(f"{silver_schema}.{silver_table_name}", 
    "Verificação de receitas não numéricas", 
    df_financeiro_filmes,
    isNumerical("receita_usd")
)

[FAIL] atividade_2.silver.tb_financeiro_filmes | Verificação de orcamentos não numéricos | 5853/99006 linhas falharam
[FAIL] atividade_2.silver.tb_financeiro_filmes | Verificação de receitas não numéricas | 8770/99006 linhas falharam


In [0]:
# try_cast trata valores não-numéricos como nulos
def cleans_numerical(column):
    return (
        F.regexp_replace(F.col(column), r"\D", "")
            .try_cast("decimal(18,2)")
    )

df_financeiro_filmes = (
    df_financeiro_filmes
        .withColumn("orcamento_usd", cleans_numerical("orcamento_usd"))
        .withColumn("receita_usd", cleans_numerical("receita_usd"))
)

df_financeiro_filmes = (
    df_financeiro_filmes
        .withColumn("orcamento_usd",
            F.when(F.col("orcamento_usd") <= 0, None)
                .otherwise(F.col("orcamento_usd")))
        .withColumn("receita_usd",
            F.when(F.col("receita_usd") <= 0, None)
                .otherwise(F.col("receita_usd")))
)

display(df_financeiro_filmes.limit(5))

id_filme,orcamento_usd,receita_usd,ingestion_datetime
293660,58000000.00,null,2026-10-16T16:18:41.150Z
299536,300000000.00,2052415039.00,2026-10-16T16:18:41.150Z
299534,356000000.00,2800000000.00,2026-10-16T16:18:41.150Z
475557,55000000.00,1074458282.00,2026-10-16T16:18:41.150Z
271110,250000000.00,null,2026-10-16T16:18:41.150Z


In [0]:
today = dbutils.widgets.get("today")

api_url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarDia(dataCotacao=@dataCotacao)?"
    f"%40dataCotacao='{today}'"
    "&%24format=json"
)

response = requests.get(api_url)

if response.status_code == 200:
    dollar_today = response.json()["value"][0]["cotacaoVenda"]
else:
    dollar_today = 5.03

df_financeiro_filmes = (
    df_financeiro_filmes.withColumns({
        "orcamento_brl": (F.col("orcamento_usd") * dollar_today).cast("decimal(18,2)"),
        "receita_brl": (F.col("receita_usd") * dollar_today).cast("decimal(18,2)")
    })
)

display(df_financeiro_filmes.limit(5))

id_filme,orcamento_usd,receita_usd,ingestion_datetime,orcamento_brl,receita_brl
293660,58000000.00,null,2026-10-16T16:18:41.150Z,289623000.00,null
299536,300000000.00,2052415039.00,2026-10-16T16:18:41.150Z,1498050000.00,10248734497.25
299534,356000000.00,2800000000.00,2026-10-16T16:18:41.150Z,1777686000.00,13981800000.00
475557,55000000.00,1074458282.00,2026-10-16T16:18:41.150Z,274642500.00,5365307431.17
271110,250000000.00,null,2026-10-16T16:18:41.150Z,1248375000.00,null


In [0]:
df_financeiro_filmes = (
    df_financeiro_filmes.withColumns(
        {
            "lucro_usd": F.try_subtract(F.col("receita_usd"), F.col("orcamento_usd")),
            "lucro_brl": F.try_subtract(F.col("receita_brl"), F.col("orcamento_brl")),
            "margem_lucro": F.try_divide(F.col("lucro_brl"), F.col("orcamento_brl")) * 100
        }
    )
)

display(df_financeiro_filmes.limit(5))

id_filme,orcamento_usd,receita_usd,ingestion_datetime,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro
293660,58000000.00,null,2026-10-16T16:18:41.150Z,289623000.00,null,null,null,null
299536,300000000.00,2052415039.00,2026-10-16T16:18:41.150Z,1498050000.00,10248734497.25,1752415039.00,8750684497.25,584.138346333566970
299534,356000000.00,2800000000.00,2026-10-16T16:18:41.150Z,1777686000.00,13981800000.00,2444000000.00,12204114000.00,686.516853932584270
475557,55000000.00,1074458282.00,2026-10-16T16:18:41.150Z,274642500.00,5365307431.17,1019458282.00,5090664931.17,1853.560512728365056
271110,250000000.00,null,2026-10-16T16:18:41.150Z,1248375000.00,null,null,null,null


In [0]:
df_financeiro_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Metricas Engajamento

In [0]:
# Mapeamento tb_movies_metrics -> tb_metricas_engajamento
silver_table_name = "tb_metricas_engajamento"
df_metricas_engajamento = spark.table(f"{bronze_schema}.tb_movies_metrics")

df_metricas_engajamento = (
    df_metricas_engajamento
        .where(F.col("id").isNotNull())
        .dropDuplicates(["id"])
        .withColumnsRenamed({
            "id": "id_filme",
            "popularity": "popularidade",
            "vote_average": "nota_media_tmdb",
            "vote_count": "qtd_votos_tmdb",
            "averageRating": "nota_media_imdb",
            "numVotes": "qtd_votos_imdb"
        })
)

display(df_metricas_engajamento.limit(5))

id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-16T16:18:42.916Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-16T16:18:42.916Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-16T16:18:42.916Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-16T16:18:42.916Z
271110,70.741,7.4,21541,7.8,947222,2026-10-16T16:18:42.916Z


In [0]:
# Limpeza de dados de popularidade

popularity_type_check = F.regexp(
    F.col("popularidade"), 
    F.lit(r"^(\d+\.{0,1}\d+|\d+\,{0,1}\d+|\d+)$")
)

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de popularidades não numéricas", 
    df_metricas_engajamento,
    popularity_type_check
)

df_corrupted_popularity = df_metricas_engajamento.filter(~popularity_type_check)

display(df_corrupted_popularity.limit(5))

[FAIL] atividade_2.silver.tb_metricas_engajamento | Verificação de popularidades não numéricas | 4167/99013 linhas falharam


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_datetime
324857,"causing others from across the Spider-Verse to be inadvertently transported to his dimension.""",8.404,null,"Phil Lord""","Rodney Rothman""""",2026-10-16T16:18:42.916Z
301528,"a road trip adventure alongside old and new friends will show Woody how big the world can be for a toy.""",7.5,9093,"John Lasseter""",Andrew Stanton,2026-10-16T16:18:42.916Z
615457,"Hutch's unknown long-simmering rage is ignited and propels him on a brutal path that will uncover dark secrets he fought to leave behind,""",8.018,6217,Derek Kolstad,7.4,2026-10-16T16:18:42.916Z
390043,"28""","214""""",6.891,5732,6.9,2026-10-16T16:18:42.916Z
140300,Po must face two hugely epic,6.878,5327,"Alessandro Carloni, Jennifer Yuh Nelson","Jonathan Aibel, Glenn Berger, Ethan Reiff, Cyrus Voris",2026-10-16T16:18:42.916Z


In [0]:
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("popularidade",
                    F.when(
                        popularity_type_check, 
                        F.regexp_replace(
                            F.col("popularidade"),
                            r"\,",
                            "."
                        ).cast("double")
                    ).otherwise(None)
        ).withColumn("popularidade",
            F.when(F.col("popularidade") <= 0, None)
                .otherwise(F.col("popularidade"))
        )
)

display(df_metricas_engajamento.limit(5))

id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-16T16:18:42.916Z
299536,154.34,8.255,27713,8.4,1406782,2026-10-16T16:18:42.916Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-16T16:18:42.916Z
475557,54.522,8.168,23425,8.3,1723035,2026-10-16T16:18:42.916Z
271110,70.741,7.4,21541,7.8,947222,2026-10-16T16:18:42.916Z


In [0]:
# Limpeza de dados de notas médias

# Substituição de vírgula por ponto
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("nota_media_tmdb",
            F.regexp_replace(F.col("nota_media_tmdb"), r"\,", ".")
                .try_cast("double")
        ).withColumn("nota_media_imdb",
            F.regexp_replace(F.col("nota_media_imdb"), r"\,", ".")
                .try_cast("double")
        )
)

# Correção de limites de notas
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("nota_media_tmdb",
            F.when(F.col("nota_media_tmdb") > 10, None)
                .when(F.col("nota_media_tmdb") < 0, None)
                .otherwise(F.col("nota_media_tmdb"))
        ).withColumn("nota_media_imdb",
            F.when(F.col("nota_media_imdb") > 10, None)
                .when(F.col("nota_media_imdb") < 0, None)
                .otherwise(F.col("nota_media_imdb"))
        )
)

display(df_metricas_engajamento.orderBy(F.desc("nota_media_imdb")))

id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_datetime
985620,0.6,0.0,0,10.0,6,2026-10-16T16:18:42.916Z
927357,0.6,0.0,0,10.0,null,2026-10-16T16:18:42.916Z
381027,null,null,7.7,10.0,7.0,2026-10-16T16:18:42.916Z
1156687,0.6,10.0,1,10.0,6,2026-10-16T16:18:42.916Z
1094301,0.6,0.0,0,10.0,10,2026-10-16T16:18:42.916Z
1096596,0.6,0.0,0,10.0,8,2026-10-16T16:18:42.916Z
1057986,0.6,0.0,0,10.0,null,2026-10-16T16:18:42.916Z
415714,null,null,6.25,10.0,6.3,2026-10-16T16:18:42.916Z
703008,0.6,null,3,10.0,8,2026-10-16T16:18:42.916Z
395802,null,null,7.6,10.0,7.5,2026-10-16T16:18:42.916Z


In [0]:
# Verificação de dados não numéricos de contagem de votos
int_check = lambda column: F.regexp(F.regexp_replace(F.col(column), r"[^\d]", ""), F.lit(r"^(\d+)$"))

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de contagem de votos não numéricos", 
    df_metricas_engajamento,
    int_check("qtd_votos_tmdb")
)

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de contagem de votos não numéricos", 
    df_metricas_engajamento,
    int_check("qtd_votos_imdb")
)

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de contagem de votos não nulos", 
    df_metricas_engajamento,
    F.col("qtd_votos_tmdb").isNotNull() & F.col("qtd_votos_imdb").isNotNull()
)

df_corrupted_count = (
    df_metricas_engajamento
        .select("id_filme", "qtd_votos_imdb", "qtd_votos_tmdb")
        .where(~int_check("qtd_votos_tmdb") | ~int_check("qtd_votos_imdb"))
)

display(df_corrupted_count.limit(5))

[FAIL] atividade_2.silver.tb_metricas_engajamento | Verificação de contagem de votos não numéricos | 15/99013 linhas falharam
[FAIL] atividade_2.silver.tb_metricas_engajamento | Verificação de contagem de votos não numéricos | 2015/99013 linhas falharam
[FAIL] atividade_2.silver.tb_metricas_engajamento | Verificação de contagem de votos não nulos | 15524/99013 linhas falharam


id_filme,qtd_votos_imdb,qtd_votos_tmdb
324857,"Rodney Rothman""""",null
301528,Andrew Stanton,9093
140300,"Jonathan Aibel, Glenn Berger, Ethan Reiff, Cyrus Voris",5327
364689,"Robert L. Baird, Tim Federle, Brad Copeland, Ron Burch, David Kidd, Don Rhymer, Munro Leaf, Robert Lawson",2652
594767,"Henry Gayden, Chris Morgan, Bill Parker, C.C. Beck, William Moulton Marston",2416


In [0]:
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("qtd_votos_tmdb",
            F.col("qtd_votos_tmdb").try_cast("int")
        ).withColumn("qtd_votos_imdb",
            F.col("qtd_votos_imdb").try_cast("int")
        )
)

# Limpeza de valores inválidos
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("qtd_votos_tmdb",
            F.when(F.col("qtd_votos_tmdb") < 0, None)
                .otherwise(F.col("qtd_votos_tmdb"))
        ).withColumn("qtd_votos_imdb",
            F.when(F.col("qtd_votos_imdb") < 0, None)
                .otherwise(F.col("qtd_votos_imdb"))
        )
)

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de contagem de votos não numéricos",
    df_metricas_engajamento,
    F.col("qtd_votos_tmdb").isNotNull() & F.col("qtd_votos_imdb").isNotNull()
)

display(df_metricas_engajamento.limit(5))

[FAIL] atividade_2.silver.tb_metricas_engajamento | Verificação de contagem de votos não numéricos | 18611/99013 linhas falharam


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-16T16:18:42.916Z
299536,154.34,8.255,27713,8.4,1406782,2026-10-16T16:18:42.916Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-16T16:18:42.916Z
475557,54.522,8.168,23425,8.3,1723035,2026-10-16T16:18:42.916Z
271110,70.741,7.4,21541,7.8,947222,2026-10-16T16:18:42.916Z


In [0]:
df_metricas_engajamento.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Avaliações usuários

In [0]:
# Mapeamento tb_movies_reviews -> tb_avaliacoes_usuarios
silver_table_name = "tb_avaliacoes_usuarios"
df_avaliacoes_usuarios = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_avaliacoes_usuarios = (
    df_avaliacoes_usuarios
        .where(F.col("id").isNotNull())
        .dropDuplicates()
        .withColumnsRenamed({
            "id": "id_filme",
            "nome": "nome_usuario",
            "nota": "nota_usuario",
            "comentario": "comentario_usuario"
        })
)

display(df_avaliacoes_usuarios.limit(5))

id_filme,nome_usuario,nota_usuario,comentario_usuario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-16T16:18:47.096Z
637007,Lucas Reis 602,3.9,null,2026-10-16T16:18:47.096Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-16T16:18:47.096Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-16T16:18:47.096Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-16T16:18:47.096Z


In [0]:
# Limpeza de dados de nota do usuário

is_numerical_check = F.regexp(F.col("nota_usuario"), F.lit(r"(\d+\.\d+|\d+)"))

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de nota de usuário não numérica", 
    df_avaliacoes_usuarios,
    is_numerical_check
)

[PASS] atividade_2.silver.tb_avaliacoes_usuarios | Verificação de nota de usuário não numérica | 0/291708 linhas falharam


In [0]:
df_avaliacoes_usuarios = (
    df_avaliacoes_usuarios
        .withColumn("nota_usuario",
            F.when(F.col("nota_usuario") > 10, None)
                .when(F.col("nota_usuario") < 0, None)
                .otherwise(F.col("nota_usuario"))
        )
)

display(df_avaliacoes_usuarios.limit(5))

id_filme,nome_usuario,nota_usuario,comentario_usuario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-16T16:18:47.096Z
637007,Lucas Reis 602,3.9,null,2026-10-16T16:18:47.096Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-16T16:18:47.096Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-16T16:18:47.096Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-16T16:18:47.096Z


In [0]:
# Padronização dos comentários

df_avaliacoes_usuarios = (
    df_avaliacoes_usuarios
        .withColumn("comentario_usuario",
            F.regexp_replace(F.col("comentario_usuario"), r"^\s*$", "Sem comentário")
        ).withColumn("comentario_usuario",
            F.when(F.col("comentario_usuario").isNull(), "Sem comentário")
                .otherwise(F.col("comentario_usuario"))
        )
)

df_no_comment = (
    df_avaliacoes_usuarios.where(F.col("comentario_usuario") == "Sem comentário")
)

display(df_no_comment.limit(5))

id_filme,nome_usuario,nota_usuario,comentario_usuario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,Sem comentário,2026-10-16T16:18:47.096Z
637007,Lucas Reis 602,3.9,Sem comentário,2026-10-16T16:18:47.096Z
413036,Gabriela Monteiro 401,7.5,Sem comentário,2026-10-16T16:18:47.096Z
739810,João Lopes 193,0.0,Sem comentário,2026-10-16T16:18:47.096Z
685997,Bruno Alves 156,5.6,Sem comentário,2026-10-16T16:18:47.096Z


In [0]:
df_avaliacoes_usuarios.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Gêneros

In [0]:
# Mapeamento tb_credits_and_tags, coluna genres -> tb_generos
silver_table_name = "tb_generos"
df_generos = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_generos = (
    df_generos.select("id", "genres")
        .dropDuplicates()
        .withColumnsRenamed({
            "id": "id_filme",
            "genres": "genero"
        })
        .withColumn("genero",
            F.explode(F.split(F.col("genero"), r"[\,\;\|]"))
        )
        .withColumn("genero",
            F.trim(F.regexp_replace(F.col("genero"), r"[^\w ]", "")))
)

display(df_generos.limit(5))

id_filme,genero
209112,Action
209112,Adventure
209112,Fantasy
181812,Adventure
181812,Action


In [0]:
df_count_generos = (
    df_generos
        .withColumn("genero", F.upper(F.col("genero")))
        .groupBy("genero")
        .count()
)

display(df_count_generos)

genero,count
SOME PRAYERS ARENT MEANT TO BE ANSWERED,1
AND THAT HE WILL NOT HESITATE TO TAKE THE NECESSARY RISKS TO ACHIEVE IT,1
WESTERN,419
A BILLIE EILISH CONCERT EXPERIENCE,1
2611,1
AND HE STARTS TO HAVE FAITH IN THE POWER OF THE LORD,1
THIS SHORT WILL EXPLORE THE THEMES OF GRIEF,1
LIJCFYSYWZ6KMPKJITF6SAZBRL3JPG,1
FCK THERAPY,1
HIS WIFE,1


In [0]:
# seleção dos gêneros com ao menos 1000 registros

window = Window.partitionBy("genero")

df_generos = (
    df_generos
        .withColumn("count_genero", F.count(F.upper(F.col("genero"))).over(window))
        .where(F.col("count_genero") >= 200)
        .drop("count_genero")
)

display(df_generos.select("genero").distinct())

genero
Action
Adventure
Animation
Comedy
Crime
Documentary
Drama
Family
Fantasy
History


In [0]:
df_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Pessoas Empresas

In [0]:
# Mapeamento  tb_credits_and_tags, colunas cast, directors, writers, production_companies -> tb_pessoas_empresas
silver_table_name = "tb_pessoas_empresas"
df_pessoas_empresas = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_pessoas_empresas = (
    df_pessoas_empresas.select("id", "cast", "directors", "writers", "production_companies")
        .withColumnsRenamed({
            "id": "id_filme",
        })
)

display(df_pessoas_empresas.limit(5))

id_filme,cast,directors,writers,production_companies
293660,"Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",Tim Miller,"Rhett Reese, Paul Wernick","20th Century Fox, The Donners' Company, Genre Films"
299536,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman","Anthony Russo, Joe Russo",N/A,Marvel Studios
299534,"Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck",Marvel Studios
475557,"Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films"
271110,"Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby",Marvel Studios


In [0]:
df_atores = (
    df_pessoas_empresas
        .select(
            "id_filme", 
            F.explode(F.split(F.upper(F.col("cast")), r"[\,\;\|]\s*")).alias("entidade"), 
            F.lit("Ator").alias("tipo_entidade")
        ).where(F.length(F.col("entidade")) > 0)
)

display(df_atores.limit(5))

id_filme,entidade,tipo_entidade
293660,RYAN REYNOLDS,Ator
293660,MORENA BACCARIN,Ator
293660,ED SKREIN,Ator
293660,T.J. MILLER,Ator
293660,GINA CARANO,Ator


In [0]:
df_diretores = (
    df_pessoas_empresas
        .select(
            "id_filme", 
            F.explode(F.split(F.upper(F.col("directors")), r"[\,\;\|]\s*")).alias("entidade"), 
            F.lit("Diretor").alias("tipo_entidade")
        ).where(F.length(F.col("entidade")) > 0)
)

display(df_diretores.limit(5))

id_filme,entidade,tipo_entidade
293660,TIM MILLER,Diretor
299536,ANTHONY RUSSO,Diretor
299536,JOE RUSSO,Diretor
299534,ANTHONY RUSSO,Diretor
299534,JOE RUSSO,Diretor


In [0]:
df_roteiristas = (
    df_pessoas_empresas
        .select(
            "id_filme", 
            F.explode(F.split(F.upper(F.col("writers")), r"[\,\;\|]\s*")).alias("entidade"), 
            F.lit("Roteirista").alias("tipo_entidade")
        ).where(F.length(F.col("entidade")) > 0)
)

display(df_roteiristas.limit(5))

id_filme,entidade,tipo_entidade
293660,RHETT REESE,Roteirista
293660,PAUL WERNICK,Roteirista
299536,N/A,Roteirista
299534,CHRISTOPHER MARKUS,Roteirista
299534,STEPHEN MCFEELY,Roteirista


In [0]:
df_produtoras = (
    df_pessoas_empresas
        .select(
            "id_filme", 
            F.explode(F.split(F.upper(F.col("production_companies")), r"[\,\;\|]\s*")).alias("entidade"), 
            F.lit("Produtora").alias("tipo_entidade")
        ).where(F.length(F.col("entidade")) > 0)
)

display(df_produtoras.limit(5))

id_filme,entidade,tipo_entidade
293660,20TH CENTURY FOX,Produtora
293660,THE DONNERS' COMPANY,Produtora
293660,GENRE FILMS,Produtora
299536,MARVEL STUDIOS,Produtora
299534,MARVEL STUDIOS,Produtora


In [0]:
df_pessoas_empresas = (
    df_atores
        .union(df_diretores)
        .union(df_roteiristas)
        .union(df_produtoras)
        .dropDuplicates()
)

display(df_pessoas_empresas.groupBy("tipo_entidade").count())

tipo_entidade,count
Produtora,120777
Roteirista,138780
Diretor,109215
Ator,549214


In [0]:
(
    df_pessoas_empresas
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.{silver_table_name}")
)

## Cotação Dólar

In [0]:
# Mapeamento tb_cotacao_dolar -> tb_cotacao_dolar
silver_table_name = "tb_cotacao_dolar"
df_cotacao_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_cotacao_dolar = (
    df_cotacao_dolar
        .withColumnRenamed("cotacaoCompra", "cotacao_compra")
        .withColumnRenamed("cotacaoVenda", "cotacao_venda")
        .withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
        .drop("dataHoraCotacao")
        .dropDuplicates(["data_cotacao"])
        .orderBy("data_cotacao")
)

display(df_cotacao_dolar.limit(7))

cotacao_compra,cotacao_venda,ingestion_datetime,data_cotacao
5.1803,5.1809,2026-10-16T15:55:59.638Z,2026-09-30
5.2073,5.2079,2026-10-16T15:55:59.638Z,2026-10-01
5.2232,5.2238,2026-10-16T15:55:59.638Z,2026-10-02
4.9853,4.9859,2026-10-16T15:55:59.638Z,2026-10-05
4.9692,4.9698,2026-10-16T15:55:59.638Z,2026-10-06
4.9929,4.9935,2026-10-16T15:55:59.638Z,2026-10-07


In [0]:
# Reindexa a tabela com todos os dias entre a primeira data e hoje
min_date = df_cotacao_dolar.select(F.min("data_cotacao")).collect()[0][0]
today = datetime.today().strftime("%Y-%m-%d")

new_idx = pd.date_range(start=min_date, end=today, freq="D")

pandas_df = df_cotacao_dolar.toPandas()

pandas_df = (
    pandas_df.set_index('data_cotacao')
        .sort_index()
        .reindex(new_idx, method="pad")
        .reset_index(names="data_cotacao")
)

df_cotacao_dolar = (
    spark.createDataFrame(pandas_df)
        .withColumn("data_cotacao",
            F.to_date(F.col("data_cotacao"), "yyyy-MM-dd")
        )
)

display(df_cotacao_dolar.limit(7))

data_cotacao,cotacao_compra,cotacao_venda,ingestion_datetime
2026-09-30,5.1803,5.1809,2026-10-16T15:55:59.638Z
2026-10-01,5.2073,5.2079,2026-10-16T15:55:59.638Z
2026-10-02,5.2232,5.2238,2026-10-16T15:55:59.638Z
2026-10-03,5.2232,5.2238,2026-10-16T15:55:59.638Z
2026-10-04,5.2232,5.2238,2026-10-16T15:55:59.638Z
2026-10-05,4.9853,4.9859,2026-10-16T15:55:59.638Z
2026-10-06,4.9692,4.9698,2026-10-16T15:55:59.638Z


In [0]:
df_cotacao_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")